# CNN Phishing Screenshot Classifier — PhishingCNN Training

Trains the custom `PhishingCNN` (4 conv blocks + MLP) to classify screenshots as
**phishing** or **legitimate** web pages.

**Output**: `model.pth` (state_dict checkpoint) compatible with `BrandClassifier`.

**Target**: F1 >= 0.93

---

### Architecture
```
Conv2d(3→32) → BN → ReLU → MaxPool
Conv2d(32→64) → BN → ReLU → MaxPool
Conv2d(64→128) → BN → ReLU → MaxPool
Conv2d(128→256) → BN → ReLU → AdaptiveAvgPool(7,7)
Flatten → Dropout(0.5) → Linear(12544→512) → ReLU → Dropout(0.3) → Linear(512→2)
```

### Instructions
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run all cells
3. Download `model.pth`, place in `backend/ml-services/visual-service/models/brand-classifier/`

In [ ]:
!pip install -q torch torchvision scikit-learn datasets Pillow

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from PIL import Image

SEED = 42
OUTPUT_DIR = "/content/brand-classifier"
EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
IMG_SIZE = 224

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### Hugging Face login (optional)

Add `HF_TOKEN` in Colab Secrets (key icon → Secrets) to load screenshot datasets from the Hub. Then run the cell below.

In [ ]:
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set. Add in Colab Secrets to load Hub datasets.")

## 1. Define Model (must match service exactly)

In [ ]:
class PhishingCNN(nn.Module):
    """Must match backend/ml-services/visual-service/src/models/cnn_classifier.py exactly."""

    def __init__(self, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((7, 7)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 7 * 7, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


model = PhishingCNN(num_classes=2).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2. Load Dataset

**Option A**: Load from HuggingFace phishing screenshot datasets.  
**Option B**: Upload your own screenshots (phishing/ and legitimate/ directories).

In [ ]:
class HuggingFaceScreenshotDataset(Dataset):
    """Load phishing screenshot dataset from HuggingFace."""

    def __init__(self, transform=None, max_samples=20000):
        self.transform = transform
        self.images = []
        self.labels = []

        from datasets import load_dataset

        datasets_to_try = [
            "phishintention/phishing-screenshot",
            "biglab/webphish",
            "Docta-ai/Phishing-Website-Screenshot",
        ]

        for ds_name in datasets_to_try:
            try:
                print(f"Loading {ds_name}...")
                ds = load_dataset(ds_name, split="train")
                count = 0
                for row in ds:
                    img = row.get("image")
                    label_raw = row.get("label", 0)

                    if img is None or not isinstance(img, Image.Image):
                        continue

                    img = img.convert("RGB")
                    label = 1 if str(label_raw).lower() in ("phishing", "1", "malicious") else int(label_raw) if isinstance(label_raw, int) else 0

                    self.images.append(img)
                    self.labels.append(label)
                    count += 1

                    if count >= max_samples:
                        break

                if len(self.images) > 100:
                    phishing = sum(self.labels)
                    print(f"  Loaded {len(self.images)} images ({phishing} phishing, {len(self.images)-phishing} legitimate)")
                    return
                else:
                    self.images.clear()
                    self.labels.clear()
            except Exception as e:
                print(f"  Could not load {ds_name}: {e}")
                self.images.clear()
                self.labels.clear()

        if not self.images:
            raise ValueError(
                "No screenshot datasets loaded. Use Option B below to upload your own screenshots."
            )

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Loading dataset from HuggingFace...")
try:
    full_dataset = HuggingFaceScreenshotDataset(transform=train_transform)
    USE_HUGGINGFACE = True
except Exception as e:
    print(f"HuggingFace datasets not available: {e}")
    print("Please use Option B below to upload your own screenshots.")
    USE_HUGGINGFACE = False

### Option B: Upload your own screenshots

Upload a zip file with structure:
```
screenshots/
  phishing/
    page1.png
    page2.jpg
  legitimate/
    page1.png
    page2.jpg
```

In [ ]:
# Uncomment to upload your own screenshots:
# from google.colab import files
# uploaded = files.upload()  # Upload a .zip file
# for filename in uploaded:
#     !unzip -q {filename} -d /content/screenshots
#
# class ImageFolderDataset(Dataset):
#     def __init__(self, image_dir, transform=None):
#         self.transform = transform
#         self.samples = []
#         for label_name, label_id in [("legitimate", 0), ("phishing", 1)]:
#             class_dir = os.path.join(image_dir, label_name)
#             if not os.path.isdir(class_dir):
#                 continue
#             for fname in os.listdir(class_dir):
#                 if fname.lower().endswith((".png", ".jpg", ".jpeg", ".webp")):
#                     self.samples.append((os.path.join(class_dir, fname), label_id))
#         print(f"Loaded {len(self.samples)} images")
#
#     def __len__(self):
#         return len(self.samples)
#
#     def __getitem__(self, idx):
#         path, label = self.samples[idx]
#         img = Image.open(path).convert("RGB")
#         if self.transform:
#             img = self.transform(img)
#         return img, label
#
# full_dataset = ImageFolderDataset("/content/screenshots", transform=train_transform)
# USE_HUGGINGFACE = True  # set to True to proceed

## 3. Create DataLoaders

In [ ]:
if USE_HUGGINGFACE or 'full_dataset' in dir():
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(
        full_dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)

    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")
else:
    print("No dataset loaded. Please load data using Option A or B above.")

## 4. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

best_f1 = 0.0
best_metrics = {}

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = torch.tensor(labels, dtype=torch.long).to(device) if not isinstance(labels, torch.Tensor) else labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    scheduler.step()

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels if isinstance(labels, list) else labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")

    print(
        f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | "
        f"Train Acc: {correct/total:.4f} | Val Acc: {acc:.4f} | Val F1: {f1:.4f}"
    )

    if f1 > best_f1:
        best_f1 = f1
        best_metrics = {
            "accuracy": float(acc), "precision": float(prec),
            "recall": float(rec), "f1": float(f1),
        }
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        # Save as state_dict checkpoint (what BrandClassifier expects)
        torch.save({
            "model_state_dict": model.state_dict(),
            "model_config": {"num_classes": 2, "input_size": IMG_SIZE},
        }, os.path.join(OUTPUT_DIR, "model.pth"))
        print(f"  -> Saved best model (F1: {best_f1:.4f})")

print(f"\nTraining complete! Best Val F1: {best_f1:.4f}")

## 5. Save Metrics & Verify

In [ ]:
# Save metrics
with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w") as f:
    json.dump(best_metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "model_card.json"), "w") as f:
    json.dump({
        "model_type": "cnn-phishing-classifier",
        "architecture": "PhishingCNN",
        "input_shape": [3, IMG_SIZE, IMG_SIZE],
        "num_classes": 2,
        "labels": {"0": "legitimate", "1": "phishing"},
        "metrics": best_metrics,
    }, f, indent=2)

# Verify model loads correctly (same way BrandClassifier does)
checkpoint = torch.load(os.path.join(OUTPUT_DIR, "model.pth"), map_location="cpu")
test_model = PhishingCNN(num_classes=checkpoint["model_config"]["num_classes"])
test_model.load_state_dict(checkpoint["model_state_dict"])
test_model.eval()

# Test with random image
dummy_img = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    output = test_model(dummy_img)
    probs = torch.softmax(output, dim=1)
print(f"\nVerification: legitimate={probs[0][0]:.3f}, phishing={probs[0][1]:.3f}")
print("Model loads and runs correctly!")

In [ ]:
# Download
!cd /content && zip -r brand-classifier.zip brand-classifier/
from google.colab import files
files.download("/content/brand-classifier.zip")